In [0]:
# ----------------------------------------------------------
# Step 0: Config - Storage Authentication
# ----------------------------------------------------------
storage_account_name = "stdataengportfolio"
storage_account_key  = "<STORAGE_ACCOUNT_KEY>"

silver_batch_path   = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/silver/superstore_historical/"
silver_api_path     = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/silver/superstore_api/"
silver_stream_path  = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/silver/superstore_stream/"
gold_path           = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/gold/sales_aggregates/"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)


In [0]:
%sql
-- 📊 Daily Revenue Trend
-- Aggregates total revenue, profit, and orders by each day.
SELECT 
    order_date,
    ROUND(SUM(daily_sales), 2) AS total_revenue,
    ROUND(SUM(daily_profit), 2) AS total_profit,
    SUM(orders_count) AS total_orders
FROM gold_sales_aggregates
WHERE order_date IS NOT NULL
GROUP BY order_date
ORDER BY order_date;


order_date,total_revenue,total_profit,total_orders
2014-01-03,null,0.2,1
2014-01-04,288.06,-65.99,3
2014-01-05,19.54,4.88,1
2014-01-06,4407.1,1358.05,9
2014-01-07,87.16,-71.96,2
2014-01-09,40.54,10.92,2
2014-01-10,54.83,22.65,2
2014-01-11,9.94,3.08,1
2014-01-13,3553.79,673.64,11
2014-01-14,61.96,-53.29,1


In [0]:
%sql
-- 🏆 Top 10 Categories by Revenue
-- Identifies highest revenue categories.
SELECT 
    category,
    ROUND(SUM(daily_sales), 2) AS total_revenue,
    ROUND(SUM(daily_profit), 2) AS total_profit
FROM gold_sales_aggregates
WHERE category IS NOT NULL
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10;


category,total_revenue,total_profit
Technology,835900.07,145388.3
Furniture,733046.86,19686.43
Office Supplies,703502.93,120632.88


In [0]:
%sql
-- 📦 Sub-Category Performance
-- Revenue and profit breakdown by category → subcategory.
SELECT 
    category,
    subcategory,
    ROUND(SUM(daily_sales), 2) AS total_revenue,
    ROUND(SUM(daily_profit), 2) AS total_profit
FROM gold_sales_aggregates
WHERE category IS NOT NULL AND subcategory IS NOT NULL
GROUP BY category, subcategory
ORDER BY category, total_revenue DESC;


category,subcategory,total_revenue,total_profit
Furniture,Chairs,328449.1,26590.17
Furniture,Tables,206965.53,-17725.48
Furniture,Bookcases,114880.0,-3472.56
Furniture,Furnishings,82752.23,14294.3
Office Supplies,Storage,216803.21,21529.91
Office Supplies,Binders,199905.72,30038.82
Office Supplies,Appliances,107532.16,18138.01
Office Supplies,Paper,75356.12,32795.56
Office Supplies,Supplies,45952.47,-1347.37
Office Supplies,Art,27118.79,6527.79


In [0]:
%sql
-- 🌍 Regional Sales Contribution
-- Shows how each region contributes to revenue, profit, and orders.
SELECT 
    region,
    ROUND(SUM(daily_sales), 2) AS total_revenue,
    ROUND(SUM(daily_profit), 2) AS total_profit,
    SUM(orders_count) AS total_orders
FROM gold_sales_aggregates
WHERE region IS NOT NULL
GROUP BY region
ORDER BY total_revenue DESC;


region,total_revenue,total_profit,total_orders
West,713471.34,107303.7,3203
East,672194.05,91603.06,2848
Central,497800.87,40150.5,2323
South,388983.59,46650.34,1620


In [0]:
%sql
-- 📈 Month-over-Month Growth
-- Aggregates sales, profit, and orders by month.
SELECT 
    DATE_FORMAT(order_date, 'yyyy-MM') AS month,
    ROUND(SUM(daily_sales), 2) AS total_revenue,
    ROUND(SUM(daily_profit), 2) AS total_profit,
    SUM(orders_count) AS total_orders
FROM gold_sales_aggregates
WHERE order_date IS NOT NULL
GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
ORDER BY month;


month,total_revenue,total_profit,total_orders
2014-01,14161.35,2417.09,79
2014-02,4119.82,786.04,46
2014-03,55526.2,419.43,157
2014-04,28139.56,3453.56,135
2014-05,23634.67,2732.58,122
2014-06,34509.0,5058.24,135
2014-07,33500.87,-925.1,143
2014-08,27603.51,5354.0,153
2014-09,81496.81,8215.78,268
2014-10,31394.94,3453.92,159


In [0]:
# --- Setup ---
from pyspark.sql import functions as F

gold_tbl = "gold_sales_aggregates"   # your Gold table name

# Base cleaned DF (no nulls)
df = (spark.table(gold_tbl)
      .where("""
        order_date IS NOT NULL AND category IS NOT NULL AND subcategory IS NOT NULL
        AND region IS NOT NULL AND daily_sales IS NOT NULL AND daily_profit IS NOT NULL
        AND orders_count IS NOT NULL
      """))

# Helper to write a single-file CSV into FileStore with a friendly name
def write_csv(df, name):
    tmp = f"dbfs:/FileStore/exports/superstore_gold/{name}_tmp"
    final = f"dbfs:/FileStore/exports/superstore_gold/{name}.csv"
    (df.coalesce(1)
       .write.mode("overwrite")
       .option("header","true")
       .csv(tmp))
    # pick the only part file and rename it
    part = [f.path for f in dbutils.fs.ls(tmp) if f.path.endswith(".csv")][0]
    dbutils.fs.mv(part, final, True)
    dbutils.fs.rm(tmp, True)

# 1) Daily revenue trend
daily_revenue = (df.groupBy("order_date")
                 .agg(F.round(F.sum("daily_sales"),2).alias("total_revenue"),
                      F.round(F.sum("daily_profit"),2).alias("total_profit"),
                      F.sum("orders_count").alias("total_orders"))
                 .orderBy("order_date"))
write_csv(daily_revenue, "daily_revenue_trend")

# 2) Top 10 categories
top_categories = (df.groupBy("category")
                  .agg(F.round(F.sum("daily_sales"),2).alias("total_revenue"),
                       F.round(F.sum("daily_profit"),2).alias("total_profit"))
                  .orderBy(F.col("total_revenue").desc())
                  .limit(10))
write_csv(top_categories, "top_categories")

# 3) Sub-category performance
subcat_perf = (df.groupBy("category","subcategory")
               .agg(F.round(F.sum("daily_sales"),2).alias("total_revenue"),
                    F.round(F.sum("daily_profit"),2).alias("total_profit"))
               .orderBy("category", F.col("total_revenue").desc()))
write_csv(subcat_perf, "subcategory_performance")

# 4) Regional contribution
regional_sales = (df.groupBy("region")
                  .agg(F.round(F.sum("daily_sales"),2).alias("total_revenue"),
                       F.round(F.sum("daily_profit"),2).alias("total_profit"),
                       F.sum("orders_count").alias("total_orders"))
                  .orderBy(F.col("total_revenue").desc()))
write_csv(regional_sales, "regional_sales")

# 5) Month-over-month growth
monthly_growth = (df.groupBy(F.date_format("order_date","yyyy-MM").alias("month"))
                  .agg(F.round(F.sum("daily_sales"),2).alias("total_revenue"),
                       F.round(F.sum("daily_profit"),2).alias("total_profit"),
                       F.sum("orders_count").alias("total_orders"))
                  .orderBy("month"))
write_csv(monthly_growth, "monthly_growth")

print("✅ CSVs written to: dbfs:/FileStore/exports/superstore_gold/")


✅ CSVs written to: dbfs:/FileStore/exports/superstore_gold/


In [0]:
%fs ls dbfs:/FileStore/exports/superstore_gold/

path,name,size,modificationTime
dbfs:/FileStore/exports/superstore_gold/daily_revenue_trend.csv,daily_revenue_trend.csv,33717,1756853704000
dbfs:/FileStore/exports/superstore_gold/monthly_growth.csv,monthly_growth.csv,1424,1756853710000
dbfs:/FileStore/exports/superstore_gold/regional_sales.csv,regional_sales.csv,167,1756853708000
dbfs:/FileStore/exports/superstore_gold/subcategory_performance.csv,subcategory_performance.csv,729,1756853707000
dbfs:/FileStore/exports/superstore_gold/top_categories.csv,top_categories.csv,131,1756853705000
